# MangoCare AI · YOLO11n training

Fine-tune pretrained YOLO11n to detect Anthracnose, Powdery Mildew, and Healthy mango leaves. Evaluate the best checkpoint on the held-out test set and export a float32 TensorFlow Lite model.

## Prepare the dataset and Colab runtime

1. On Windows, open `D:\Mango_Disease`, right-click the prepared `YOLO_3Class` folder, and choose **Compress to ZIP file** (or **Send to → Compressed (zipped) folder**). Name the archive `YOLO_3Class.zip`. It must contain `data.yaml` and the `train`, `valid`, and `test` folders, each with `images` and `labels`. The ZIP may contain a top-level `YOLO_3Class` folder.
2. In Google Drive, create a `MangoCare` folder under **My Drive** and upload `YOLO_3Class.zip` there. Keep the original Windows dataset in place.
3. [Open Google Colab](https://colab.research.google.com/), choose **File → Upload notebook**, and select this notebook. Choose **Runtime → Change runtime type → GPU**. Then choose **Runtime → Run all** when ready and authorize the Drive mount when prompted.

## Install Ultralytics and verify CUDA

The notebook uses the GPU assigned by Colab. If CUDA is unavailable, switch to a GPU runtime and rerun from the first cell.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.223"
])

import torch
import ultralytics
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU in Runtime → Change runtime type.")

torch.zeros(1, device="cuda:0").sum().item()
device = 0
gpu = torch.cuda.get_device_properties(device)
print(f"GPU: {gpu.name} · {gpu.total_memory / 1024**3:.1f} GiB")
print(f"PyTorch {torch.__version__} · Ultralytics {ultralytics.__version__}")

## Mount Drive and stage the dataset

The archive is read from Drive. The notebook extracts a fresh copy into `/content` and writes a separate Colab YAML there. Set `zip_path` below if you uploaded the archive to another folder.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile

import yaml
from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/MangoCare")
zip_path = drive_root / "YOLO_3Class.zip"
if not zip_path.is_file():
    raise FileNotFoundError(f"Upload the dataset ZIP to {zip_path} or update zip_path.")

work_dir = Path(mkdtemp(prefix="mangocare-", dir="/content"))
extracted = work_dir / "dataset"
with ZipFile(zip_path) as archive:
    for entry in archive.infolist():
        if not (extracted / entry.filename).resolve().is_relative_to(extracted):
            raise ValueError(f"Unsafe path in dataset ZIP: {entry.filename}")
    archive.extractall(extracted)

dataset_root = next(
    (folder for folder in (extracted, extracted / "YOLO_3Class")
     if (folder / "data.yaml").is_file()),
    None,
)
if dataset_root is None:
    raise FileNotFoundError("Expected data.yaml at the ZIP root or inside YOLO_3Class.")

source_config = yaml.safe_load((dataset_root / "data.yaml").read_text(encoding="utf-8"))
class_names = {0: "Anthracnose", 1: "Powdery Mildew", 2: "Healthy"}
if source_config.get("names") != class_names or source_config.get("nc") != 3:
    raise ValueError(f"Expected these class IDs in data.yaml: {class_names}")

splits = {"train": "train", "val": "valid", "test": "test"}
config = {"path": str(dataset_root), "nc": len(class_names), "names": class_names}
for key, folder in splits.items():
    images = dataset_root / folder / "images"
    labels = dataset_root / folder / "labels"
    if not images.is_dir() or not labels.is_dir():
        raise FileNotFoundError(f"Missing {key} images or labels under {dataset_root / folder}.")
    config[key] = f"{folder}/images"

data_yaml = work_dir / "data_colab.yaml"
data_yaml.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print(data_yaml.read_text(encoding="utf-8"))

## Train YOLO11n and save the best checkpoint

Train for 50 epochs at 640 × 640 with seed 42. AutoBatch profiles the allocated GPU using a 45% memory target to leave room for validation. If the run reports CUDA out of memory, set `train_batch` to `8` or `4` and rerun this cell. Training uses the validation split to select `best.pt`, then copies it to a new run folder in Drive.

In [ ]:
import shutil
from datetime import datetime, timezone
from uuid import uuid4

image_size = 640
train_batch = 0.45
model = YOLO("yolo11n.pt", task="detect")
model.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=image_size,
    batch=train_batch,
    device=device,
    seed=42,
    deterministic=True,
    cache=False,
    workers=2,
    project=str(work_dir / "runs"),
    name="yolo11n",
)

best_local = Path(model.trainer.best)
if not best_local.is_file():
    raise FileNotFoundError(f"Training did not produce a best checkpoint at {best_local}.")

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
drive_run = drive_root / "models" / "yolo_detection" / f"yolo11n-{stamp}-{uuid4().hex[:8]}"
drive_run.mkdir(parents=True)
best_drive = drive_run / "best.pt"
shutil.copy2(best_local, best_drive)
print(f"Best checkpoint saved to {best_drive}")

## Evaluate on the held-out test set

Reload the best checkpoint and evaluate it against the `test` split. Save precision, recall, mAP50, mAP50–95, and the evaluation plots to Drive.

In [ ]:
import json

best_model = YOLO(str(best_local), task="detect")
test_metrics = best_model.val(
    data=str(data_yaml),
    split="test",
    imgsz=image_size,
    batch=4,
    device=device,
    project=str(work_dir / "evaluation"),
    name="test",
    plots=True,
)
summary = {key: float(value) for key, value in test_metrics.results_dict.items()}
summary_path = drive_run / "test_metrics.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
shutil.copytree(test_metrics.save_dir, drive_run / "test_evaluation", dirs_exist_ok=True)
print(json.dumps(summary, indent=2))

## Export TensorFlow Lite for offline Android inference

Export the best checkpoint as a float32 `.tflite` with a fixed 640 × 640 input and batch size 1. Ultralytics may install conversion dependencies on the first export. Save `labels.txt` in class-ID order alongside the model. The Android app will need matching letterbox preprocessing, confidence filtering, non-maximum suppression (NMS), and bounding-box coordinate mapping.

In [ ]:
# Export from the local checkpoint so intermediate files stay in /content.
export_model = YOLO(str(best_local), task="detect")
tflite_local = Path(export_model.export(
    format="tflite",
    imgsz=image_size,
    batch=1,
    device="cpu",
    half=False,
    int8=False,
    nms=False,
))
with tflite_local.open("rb") as model_file:
    if model_file.read(8)[4:8] != b"TFL3":
        raise ValueError(f"Expected a TensorFlow Lite FlatBuffer at {tflite_local}.")

tflite_drive = drive_run / "mangocare_yolo11n_float32.tflite"
shutil.copy2(tflite_local, tflite_drive)
(drive_run / "labels.txt").write_text("\n".join(class_names.values()) + "\n", encoding="utf-8")
print(f"TensorFlow Lite model saved to {tflite_drive}")
print(f"Weights, test results, and Android export: {drive_run}")